In [5]:
# Versione 6 giugno: pipeline completa con tanto di Leave-One-Run-Out Cross Validation
# Prova di motor imagery specifica per rest/active
import mne
from mne.decoding import CSP
from mne_bids import BIDSPath, read_raw_bids
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from pathlib import Path
import warnings
import logging

warnings.filterwarnings("ignore")
mne.set_log_level('ERROR')
logging.getLogger('mne').setLevel(logging.ERROR)

root = Path("../data").resolve()
runs = ["4", "8", "12"]
test_run_order = runs.copy()

first_person = 1
people = 10
all_accuracy = np.zeros((people, len(runs)))
cm_sum = np.zeros((2, 2))

window_size   = 2.0
step_size     = 0.5
onset_offset  = 0.5   # salta i primi 0.5s dopo l'onset
task_duration = 4.0

channels_of_interest = ["C3", "C4", "Cz", "Fc3", "Fc4", "Cp3", "Cp4"]

for i in range(first_person, first_person + people):
    subject = f"{i:03d}"

    for test_index, test_run in enumerate(test_run_order):
        train_runs = [r for r in runs if r != test_run]

        X_train, y_train = [], []
        X_test,  y_test  = [], []

        for run in runs:
            bids_path = BIDSPath(
                subject=subject,
                task="motion",
                run=run,
                datatype="eeg",
                root=root,
            )

            try:
                raw = read_raw_bids(bids_path, verbose=False)
                raw.load_data(verbose=False)
                raw.filter(l_freq=8, h_freq=30, verbose=False)
                raw.set_eeg_reference('average', projection=False, verbose=False)

                events, event_id = mne.events_from_annotations(raw, verbose=False)
                event_map = {
                    event_id['TASK2T0']: 0,   # rest
                    event_id['TASK2T1']: 1,   # active (sinistra)
                    event_id['TASK2T2']: 1,   # active (destra) — aggregati
                }

                sfreq          = int(raw.info['sfreq'])
                window_samples = int(window_size   * sfreq)
                step_samples   = int(step_size     * sfreq)
                task_samples   = int(task_duration * sfreq)
                offset_samples = int(onset_offset  * sfreq)

                picks = mne.pick_channels(raw.ch_names, channels_of_interest)
                data  = raw.get_data(picks=picks)

                windows_run = []
                labels_run  = []

                for event in events:
                    onset = event[0]
                    code  = event[2]

                    if code not in event_map:
                        continue

                    label = event_map[code]

                    # Sliding window con offset iniziale
                    start = onset + offset_samples
                    while start + window_samples <= onset + task_samples:
                        end = start + window_samples
                        if end > data.shape[1]:
                            break
                        windows_run.append(data[:, start:end])
                        labels_run.append(label)
                        start += step_samples

                if not windows_run:
                    continue

                windows_run = np.array(windows_run)
                labels_run  = np.array(labels_run)

                if run in train_runs:
                    X_train.append(windows_run)
                    y_train.append(labels_run)
                else:
                    X_test.append(windows_run)
                    y_test.append(labels_run)

            except Exception as e:
                print(f"Errore soggetto {subject} run {run}: {e}")

        if not X_train or not X_test:
            continue

        X_train = np.concatenate(X_train, axis=0)
        y_train = np.concatenate(y_train)
        X_test  = np.concatenate(X_test,  axis=0)
        y_test  = np.concatenate(y_test)

        # Verifica bilanciamento
        unique, counts = np.unique(y_train, return_counts=True)
        print(f"Soggetto {subject} | Test run: {test_run}")

        pipe = Pipeline([
            ("csp",    CSP(reg='ledoit_wolf')),
            ("scaler", StandardScaler()),
            ("svm",    SVC(probability=True, class_weight='balanced'))
        ])

        param_grid = {
            "csp__n_components": [2, 4, 6],
            "csp__log":          [True],
            "svm__C":            [0.1, 1, 10, 100],
            "svm__gamma":        ["scale"],
            "svm__kernel":       ["rbf", "linear"]
        }

        grid = GridSearchCV(
            pipe, param_grid,
            cv=5,
            scoring="balanced_accuracy",
            n_jobs=-1
        )
        grid.fit(X_train, y_train)

        probs       = grid.predict_proba(X_test)
        max_probs   = np.max(probs, axis=1)
        predictions = np.argmax(probs, axis=1)

        # Threshold dinamico
        threshold        = 0.90
        min_accepted_ratio = 0.70
        accepted_mask    = max_probs >= threshold

        while np.sum(accepted_mask) < min_accepted_ratio * len(y_test) and threshold > 0.50:
            threshold -= 0.05
            accepted_mask = max_probs >= threshold

        accepted_predictions = predictions[accepted_mask]
        accepted_true_labels = y_test[accepted_mask]

        accuracy = balanced_accuracy_score(accepted_true_labels, accepted_predictions)
        cm       = confusion_matrix(accepted_true_labels, accepted_predictions)
        cm_sum  += confusion_matrix(y_test, predictions)

        accepted  = np.sum(accepted_mask)
        total     = len(y_test)
        discarded = total - accepted

        print(f"Threshold finale: {threshold:.2f}")
        print(cm)
        print(f"Campioni totali/accettati/scartati: {total}/{accepted}/{discarded}")
        print(f"Percentuale scartata: {100*discarded/total:.2f}%")
        print(f"Balanced accuracy sui campioni accettati: {accuracy*100:.2f}%")
        print("----------------------------------------------------")

        all_accuracy[i - first_person, test_index] = accuracy

    patient_mean = all_accuracy[i - first_person].mean()
    print(f"Accuratezza media paziente {subject}: {patient_mean:.4f}")
    print("====================================================")

print(f"\nAccuratezza media:   {all_accuracy.mean():.4f}")
print(f"Deviazione standard: {all_accuracy.std():.4f}")
print(f"Matrice di confusione media:\n{cm_sum / people}")

Soggetto 001 | Test run: 4
Threshold finale: 0.70
[[44  6]
 [18 23]]
Campioni totali/accettati/scartati: 120/91/29
Percentuale scartata: 24.17%
Balanced accuracy sui campioni accettati: 72.05%
----------------------------------------------------
Soggetto 001 | Test run: 8
Threshold finale: 0.60
[[29 14]
 [ 2 49]]
Campioni totali/accettati/scartati: 120/94/26
Percentuale scartata: 21.67%
Balanced accuracy sui campioni accettati: 81.76%
----------------------------------------------------
Soggetto 001 | Test run: 12
Threshold finale: 0.65
[[42  6]
 [18 23]]
Campioni totali/accettati/scartati: 120/89/31
Percentuale scartata: 25.83%
Balanced accuracy sui campioni accettati: 71.80%
----------------------------------------------------
Accuratezza media paziente 001: 0.7520
Soggetto 002 | Test run: 4
Threshold finale: 0.70
[[24 17]
 [11 37]]
Campioni totali/accettati/scartati: 120/89/31
Percentuale scartata: 25.83%
Balanced accuracy sui campioni accettati: 67.81%
-----------------------------